<a href="https://colab.research.google.com/github/engareej1/ASL-Project/blob/main/sign_language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # ← Upload your kaggle.json file here (the one you downloaded from Kaggle Settings > API)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Now download the dataset
!kaggle datasets download -d datamunge/sign-language-mnist -q
!unzip -o -q sign-language-mnist.zip
!rm sign-language-mnist.zip
print("Dataset downloaded and extracted successfully!")

Saving kaggle.json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
Dataset downloaded and extracted successfully!


In [ ]:
!pip uninstall -y numpy mediapipe tensorflow gradio -q
!pip install numpy==1.26.4 mediapipe==0.10.14 tensorflow==2.16.1 gradio==4.44.0 opencv-python-headless==4.10.0.84 -q

import os
os.kill(os.getpid(), 9)  # Auto-restart to apply changes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.16.1 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
orbax-checkpoint 0.11.36 requires jax>=0.6.0, but you have jax 0.4.34 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.16.1 which is incom

In [ ]:
!kaggle datasets download -d datamunge/sign-language-mnist -q
!unzip -o -q sign-language-mnist.zip
!rm sign-language-mnist.zip
print("Dataset downloaded and ready!")

Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
Dataset downloaded and ready!


In [ ]:
!kaggle datasets download -d datamunge/sign-language-mnist -q
!unzip -o -q sign-language-mnist.zip
!rm sign-language-mnist.zip
print("Dataset re-downloaded and extracted successfully!")

Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
Dataset re-downloaded and extracted successfully!


In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.utils import to_categorical

train = pd.read_csv('sign_mnist_train/sign_mnist_train.csv')
test = pd.read_csv('sign_mnist_test/sign_mnist_test.csv')

# Fix labels: A=0 to Y=23
y_train = train['label'].values.copy()
y_test = test['label'].values.copy()
y_train[y_train >= 9] -= 1
y_test[y_test >= 9] -= 1

X_train = train.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = test.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0

y_train = to_categorical(y_train, 24)
y_test = to_categorical(y_test, 24)

print(f"Data ready! Train: {X_train.shape}, Test: {X_test.shape}")

Data ready! Train: (27455, 28, 28, 1), Test: (7172, 28, 28, 1)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Conv2D(64, 3, activation='relu', input_shape=(28,28,1), padding='same'),
    BatchNormalization(),
    Conv2D(64, 3, activation='relu', padding='same'),
    MaxPooling2D(2),
    Dropout(0.25),

    Conv2D(128, 3, activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, 3, activation='relu', padding='same'),
    MaxPooling2D(2),
    Dropout(0.25),

    Conv2D(256, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2),
    Dropout(0.25),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(24, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

datagen = ImageDataGenerator(rotation_range=15, zoom_range=0.15, width_shift_range=0.1, height_shift_range=0.1)
datagen.fit(X_train)

model.fit(datagen.flow(X_train, y_train, batch_size=128),
          validation_data=(X_test, y_test),
          epochs=30,
          callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=1)])

model.save('asl_model.h5')
print("Model trained and saved!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.4546 - loss: 1.7920 - val_accuracy: 0.0608 - val_loss: 6.8329
Epoch 2/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 290s 1s/step - accuracy: 0.8452 - loss: 0.4479 - val_accuracy: 0.4509 - val_loss: 2.3034
Epoch 3/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 296s 1s/step - accuracy: 0.9305 - loss: 0.2096 - val_accuracy: 0.9534 - val_loss: 0.1722
Epoch 4/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 295s 1s/step - accuracy: 0.9540 - loss: 0.1370 - val_accuracy: 0.9847 - val_loss: 0.0444
Epoch 5/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.9689 - loss: 0.0955 - val_accuracy: 0.9875 - val_loss: 0.0344
Epoch 6/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - accuracy: 0.9748 - loss: 0.0730 - val_accuracy: 0.9943 - val_loss: 0.0191
Epoch 7/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 293s 1s/step - accuracy: 0.9784 - loss: 0.0661 - val_accuracy: 0.9996 - val_loss: 0.0042
Epoch 8/30
215/215 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accuracy: 0.9794 - loss: 0.0642 - val_accu

Model trained and saved!


In [ ]:
import string
import json

letters = list(string.ascii_uppercase)
letters.pop(9)   # remove J
letters.pop(24)  # remove Z
class_map = {str(i): letters[i] for i in range(24)}

with open('class_map.json', 'w') as f:
    json.dump(class_map, f)

print("Letter mapping created!")

Letter mapping created!


In [ ]:
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.3.0
    Uninstalling gradio_client-1.3.0:
      Successfully uninstalled gradio_client-1.3.0
  Attempting uninstall: gradio
    Found existing installation: gradio 4.44.0
    Uninstalling gradio-4.44.0:
      Successfully uninstalled gradio-4.44.0


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import gradio as gr
from tensorflow.keras.models import load_model
import json
import time
from collections import deque

model = load_model('asl_model.h5')
with open('class_map.json') as f:
    class_map = json.load(f)

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.8, min_tracking_confidence=0.8)

current_word = ""
pred_buffer = deque(maxlen=8)
last_add_time = 0
no_hand_start = 0

def process_frame(frame):
    global current_word, last_add_time, no_hand_start, pred_buffer

    # Make a copy to draw on
    output_frame = frame.copy()

    rgb = cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    display_text = f"Word: {current_word}" if current_word else "Start signing letters!"

    if results.multi_hand_landmarks:
        no_hand_start = 0
        hand = results.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(output_frame, hand, mp_hands.HAND_CONNECTIONS)

        x_coords = [lm.x for lm in hand.landmark]
        y_coords = [lm.y for lm in hand.landmark]
        x_min = max(0, int(min(x_coords) * frame.shape[1]) - 40)
        x_max = min(frame.shape[1], int(max(x_coords) * frame.shape[1]) + 40)
        y_min = max(0, int(min(y_coords) * frame.shape[0]) - 40)
        y_max = min(frame.shape[0], int(max(y_coords) * frame.shape[0]) + 40)

        crop = frame[y_min:y_max, x_min:x_max]
        if crop.shape[0] > 20 and crop.shape[1] > 20:
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray, (28, 28))
            img = resized.reshape(1, 28, 28, 1) / 255.0
            pred = model.predict(img, verbose=0)[0]
            idx = np.argmax(pred)
            conf = pred[idx]

            if conf > 0.9:
                pred_buffer.append(idx)
                if len(pred_buffer) == pred_buffer.maxlen:
                    stable_idx = max(set(pred_buffer), key=list(pred_buffer).count)
                    if time.time() - last_add_time > 0.8:
                        current_word += class_map[str(stable_idx)]
                        last_add_time = time.time()

    else:
        if no_hand_start == 0:
            no_hand_start = time.time()
        elapsed = time.time() - no_hand_start

        if 1.5 < elapsed < 3.5 and current_word and current_word[-1] != " ":
            current_word += " "
        elif elapsed > 5:
            current_word = ""

    # Draw text on output frame
    cv2.putText(output_frame, display_text, (30, 70), cv2.FONT_HERSHEY_DUPLEX, 2, (0, 255, 0), 4)
    cv2.putText(output_frame, "Hold letter • Short pause = space • Long pause = clear", (30, output_frame.shape[0]-30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    return output_frame  # Return the processed frame

gr.Interface(
    fn=process_frame,
    inputs=gr.Image(type="numpy", streaming=True, label="Your Hand"),
    outputs=gr.Image(type="numpy", label="Live View + Building Word"),
    title="Real-Time ASL Fingerspelling → Text",
    description="Sign letters slowly. Hold each 1-2 seconds. Short pause = space, long pause = clear.",
    live=True
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://35a41a6c948af081c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
